# Higher-order binary optimization (PUBO) in Python

Not every binary problem is quadratic. When the objective multiplies **three or
more** binary variables together, it is a **PUBO** (polynomial) or **HUBO**
(higher-order) problem — the class quantum computers are built for, here solved
classically. **quicopt** optimizes the higher-order objective **directly**: no
reduction to a quadratic QUBO, no auxiliary variables.

In [1]:
%pip install "quicopt[pyomo]"

## The naive approach

The common workaround is to reduce the degree — introduce a helper binary for
each product of variables until everything is quadratic, then feed a QUBO solver.
That inflates the variable count and adds penalty terms that reshape the energy
landscape. Solve the objective as written instead.

## A worked example: LABS

The **low-autocorrelation binary sequence (LABS)** problem asks for a string of
±1 symbols that looks as little like a shifted copy of itself as possible. Its
energy is a **degree-four** polynomial in the binaries — a genuine PUBO. Modeled
in **Pyomo** (`pip install "quicopt[pyomo]"`), here for `N = 7`:

In [2]:
import pyomo.environ as pyo
from quicopt import Client
# LABS (low-autocorrelation binary sequences) for N=7: a degree-4 (PUBO) energy.
N = 7
m = pyo.ConcreteModel()
m.x = pyo.Var(range(N), domain=pyo.Binary)
s = [1 - 2 * m.x[i] for i in range(N)]
m.obj = pyo.Objective(
    expr=sum(sum(s[i] * s[i + k] for i in range(N - k)) ** 2 for k in range(1, N)),
    sense=pyo.minimize)
print(Client().solve(m).display)

▄▀▀▀▄        ▄                     █
█   █ █   █ ▄▄  ▄▀▀▀▄ ▄▀▀▀▄ █▀▀▀▄ ▀█▀▀
█ ▀▄▀ █   █  █  █   ▄ █   █ █▄▄▄▀  █  ▄
 ▀▀ ▀  ▀▀▀▀ ▀▀▀  ▀▀▀   ▀▀▀  █       ▀▀

├── shots
│   ├── 1 · primal   91    0.0s
│   ├── 2 · primal    3   0.01s  ◀ best
│   ├── 3 · primal    3   2.54s
│   ├── 4 · primal    3    2.0s
│   └── 5 · primal    3   2.55s
├── status:     heuristic
├── feasible:   n/a
├── objective:  3.0
├── x:          x1=1, x2=0, x3=1, x4=1, x5=0, x6=0, …  (7 variables)
└── solve_time: 2.5476 s


## What you get

quicopt returns energy `3` — the **known optimum** for LABS at `N = 7` (established
by exhaustive results in the literature) — straight from the degree-four
objective, with no manual reduction to quadratic. Because a PUBO has no
constraints, the status is `heuristic` and `feasible` is `n/a`.

Read the `shots` block before you read `solve_time`. Energy `3` is on the board
almost at once; the later shots keep working the non-convex energy and land on
the same value, and that is what the two and a half seconds of wall clock pay
for. A PUBO gives
the solver no dual bound to stop on, so the search has no way to know it is
finished except to keep looking.

The same directness scales: on the public LABS instances quicopt reaches
best-known energies orders of magnitude faster than established solvers — see the
[LABS benchmark](https://quicopt.com/benchmarks/labs).

## Next

- The problem class behind this: [PUBO / higher-order binary](https://quicopt.com/problems/pubo)
- Measured results at scale: [LABS benchmark](https://quicopt.com/benchmarks/labs)
- The quadratic cousin: [Max-cut (QUBO)](https://quicopt.com/developer/tutorials/max-cut)

**Reference:** T. Packebusch and S. Mertens, *Low autocorrelation binary
sequences*, Journal of Physics A, 2016.

## Frequently asked questions

### What's the difference between QUBO and PUBO?

QUBO objectives are degree two (pairwise). PUBO (polynomial) and HUBO (higher-order) allow degree three and up — products of three or more binary variables. Many real binary objectives are naturally higher-order.

### Why not just reduce it to a QUBO?

Reducing a degree-k objective to quadratic adds auxiliary variables and penalty terms that blow up the model and distort the landscape. quicopt optimizes the higher-order objective directly, as written.

### Is quicopt free to use?

Yes — pip install quicopt and your first call sets up a free key, no license.